### 1. Retrive the raw data

In [7]:
import requests
import pandas as pd
import json
from pathlib import Path

url = "https://api.energy-charts.info/price"

params = {
    "bzn": "DE-LU",
    "start": "2025-01-01",
    "end": "2026-01-01"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

print("HTTP status:", response.status_code)
print("Available fields:", list(data.keys()))
print("Number of timestamps:", len(data["unix_seconds"]))
print("Number of prices:", len(data["price"]))

HTTP status: 200
Available fields: ['license_info', 'unix_seconds', 'price', 'unit', 'deprecated']
Number of timestamps: 15483
Number of prices: 15483


### 2. preserve the original data
The original API response is stored without modification
to ensure transparency and reproducibility.

In [8]:
# Identify the project root directory
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define raw data directory
raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Preserve the original API response
file_path = raw_dir / "DE_LU_day_ahead_2025.json"
file_path.write_bytes(response.content)

print("Raw data saved to:", file_path)
print("Price unit:", data["unit"])
print("License information:", data["license_info"])
print("Deprecated:", data["deprecated"])

Raw data saved to: /Users/laramileto/Documents/HSG/5° semestre /Ökonomie der Elektromobilität/reflection paper/Economic-Value-of-Smart-EV-Charging/data/raw/DE_LU_day_ahead_2025.json
Price unit: EUR / MWh
License information: CC BY 4.0 (creativecommons.org/licenses/by/4.0) from Bundesnetzagentur | SMARD.de
Deprecated: False


### 3. Convert the data into a DataFrame
Convert Unix timestamps into timezone-aware datetime
objects, filter the observations to the calendar year
2025, and convert electricity prices into EUR/kWh.

The Europe/Berlin timezone is used to preserve the
German electricity market's local time, including
daylight-saving transitions.

In [10]:
# Create DataFrame
df = pd.DataFrame({
    "timestamp_utc": pd.to_datetime(
        data["unix_seconds"],
        unit="s",
        utc=True
    ),
    "price_eur_mwh": data["price"]
})

# Convert timestamps to German local time
df["timestamp_local"] = (
    df["timestamp_utc"].dt.tz_convert("Europe/Berlin")
)

# Inspect the original temporal coverage
print("Original number of observations:", len(df))
print("Original first timestamp:", df["timestamp_local"].min())
print("Original last timestamp:", df["timestamp_local"].max())

# Restrict the dataset to 2025
df = df[df["timestamp_local"].dt.year == 2025].copy()

# Sort observations chronologically
df = df.sort_values("timestamp_utc").reset_index(drop=True)

# Convert electricity prices from EUR/MWh to EUR/kWh
df["price_eur_kwh"] = df["price_eur_mwh"] / 1000

# Inspect filtered dataset
print("\nFiltered number of observations:", len(df))
print("First timestamp:", df["timestamp_local"].min())
print("Last timestamp:", df["timestamp_local"].max())

display(df.head(10))

Original number of observations: 15483
Original first timestamp: 2025-01-01 00:00:00+01:00
Original last timestamp: 2026-01-01 23:45:00+01:00

Filtered number of observations: 15387
First timestamp: 2025-01-01 00:00:00+01:00
Last timestamp: 2025-12-31 23:45:00+01:00


,timestamp_utc,price_eur_mwh,timestamp_local,price_eur_kwh
0,2024-12-31 23:00:00+00:00,2.16,2025-01-01 00:00:00+01:00,0.00216
1,2025-01-01 00:00:00+00:00,1.60,2025-01-01 01:00:00+01:00,0.00160
2,2025-01-01 01:00:00+00:00,0.00,2025-01-01 02:00:00+01:00,0.00000
3,2025-01-01 02:00:00+00:00,-0.01,2025-01-01 03:00:00+01:00,-0.00001
4,2025-01-01 03:00:00+00:00,-0.01,2025-01-01 04:00:00+01:00,-0.00001
5,2025-01-01 04:00:00+00:00,-0.06,2025-01-01 05:00:00+01:00,-0.00006
6,2025-01-01 05:00:00+00:00,-0.09,2025-01-01 06:00:00+01:00,-0.00009
7,2025-01-01 06:00:00+00:00,-0.01,2025-01-01 07:00:00+01:00,-0.00001
8,2025-01-01 07:00:00+00:00,-0.01,2025-01-01 08:00:00+01:00,-0.00001
9,2025-01-01 08:00:00+00:00,-0.04,2025-01-01 09:00:00+01:00,-0.00004


### 4. Conduct an initial data quality audit

Before conducting the economic analysis, the dataset is
examined for missing values, duplicate timestamps,
invalid numerical observations and inconsistencies
in temporal resolution.

Negative electricity prices are retained because they
represent economically meaningful market observations
rather than data errors.

In [12]:
import numpy as np


# 1. Dataset dimensions
print("\n1. Dataset dimensions")
print("Observations:", len(df))
print("Variables:", len(df.columns))

# 2. Missing values
print("\n2. Missing values")
print(df.isna().sum())

# 3. Duplicate timestamps
print("\n3. Duplicate UTC timestamps")
print(df["timestamp_utc"].duplicated().sum())

# 4. Invalid numerical values
print("\n4. Non-finite electricity prices")
print((~np.isfinite(df["price_eur_mwh"])).sum())

# 5. Chronological consistency
print("\n5. Chronological order")
print(df["timestamp_utc"].is_monotonic_increasing)

# 6. Temporal resolution
print("\n6. Time intervals between observations")
print(df["timestamp_utc"].diff().value_counts())

# 7. Preliminary descriptive statistics
print("\n7. Electricity price statistics (EUR/MWh)")
print(df["price_eur_mwh"].describe())


1. Dataset dimensions
Observations: 15387
Variables: 4

2. Missing values
timestamp_utc      0
price_eur_mwh      0
timestamp_local    0
price_eur_kwh      0
dtype: int64

3. Duplicate UTC timestamps
0

4. Non-finite electricity prices
0

5. Chronological order
True

6. Time intervals between observations
timestamp_utc
0 days 00:15:00    8835
0 days 01:00:00    6551
Name: count, dtype: int64

7. Electricity price statistics (EUR/MWh)
count    15387.000000
mean        90.971903
std         49.851455
min       -250.320000
25%         73.485000
50%         90.810000
75%        111.010000
max        583.400000
Name: price_eur_mwh, dtype: float64


### 5. Check for missing timestamps

The European day-ahead electricity market transitioned
from hourly to 15-minute market time units on
1 October 2025.

The expected timestamp index is constructed separately
for both periods, accounting for daylight-saving
transitions in the Europe/Berlin timezone.

Actual timestamps are compared against the expected
index to identify missing or unexpected observations.

In [14]:
# Define relevant dates
start = pd.Timestamp(
    "2025-01-01",
    tz="Europe/Berlin"
)

transition = pd.Timestamp(
    "2025-10-01",
    tz="Europe/Berlin"
)

end = pd.Timestamp(
    "2026-01-01",
    tz="Europe/Berlin"
)

# Expected hourly observations: January–September
expected_hourly = pd.date_range(
    start=start,
    end=transition,
    freq="h",
    inclusive="left"
)

# Expected quarter-hourly observations: October–December
expected_quarterly = pd.date_range(
    start=transition,
    end=end,
    freq="15min",
    inclusive="left"
)

# Combine both periods and convert to UTC
expected = expected_hourly.append(
    expected_quarterly
).tz_convert("UTC")

# Actual timestamps
actual = pd.DatetimeIndex(df["timestamp_utc"])

# Identify inconsistencies
missing = expected.difference(actual)
unexpected = actual.difference(expected)


print("\nExpected hourly observations:", len(expected_hourly))
print("Expected quarterly observations:", len(expected_quarterly))

print("\nTotal expected observations:", len(expected))
print("Total actual observations:", len(actual))

print("\nMissing timestamps:", len(missing))
print("Unexpected timestamps:", len(unexpected))

print("\nFirst missing timestamps:")
print(missing[:10])

print("\nFirst unexpected timestamps:")
print(unexpected[:10])

# Calculate timestamp coverage
coverage = len(expected.intersection(actual)) / len(expected) * 100

print(f"\nTemporal coverage: {coverage:.2f}%")


Expected hourly observations: 6551
Expected quarterly observations: 8836

Total expected observations: 15387
Total actual observations: 15387

Missing timestamps: 0
Unexpected timestamps: 0

First missing timestamps:
DatetimeIndex([], dtype='datetime64[us, UTC]', freq=None)

First unexpected timestamps:
DatetimeIndex([], dtype='datetime64[us, UTC]', freq=None)

Temporal coverage: 100.00%


### 6. Temporal Harmonisation

The original dataset contains hourly electricity prices
from January to September 2025 and quarter-hourly prices
from October to December 2025.

To ensure temporal consistency and prevent disproportionate
weighting of quarter-hourly observations, all prices are
converted to a common hourly resolution.

Quarter-hourly prices are aggregated using their arithmetic
mean, representing the hourly price under the assumption
of constant electricity consumption within each hour.

Aggregation is performed in UTC to avoid ambiguity during
daylight-saving transitions. Local German time is retained
for the subsequent analysis of charging behaviour.

No missing observations are interpolated, and negative
electricity prices are preserved.

In [15]:
# Set UTC timestamps as the index
prices = df.set_index("timestamp_utc")["price_eur_mwh"]

# Aggregate observations to hourly resolution
df_hourly = (
    prices
    .resample("h")
    .mean()
    .to_frame()
    .reset_index()
)

# Count original observations contributing to each hour
df_hourly["source_observations"] = (
    prices.resample("h").count().to_numpy()
)

# Convert timestamps to German local time
df_hourly["timestamp_local"] = (
    df_hourly["timestamp_utc"]
    .dt.tz_convert("Europe/Berlin")
)

# Convert prices into EUR/kWh
df_hourly["price_eur_kwh"] = (
    df_hourly["price_eur_mwh"] / 1000
)

# Inspect the harmonised dataset
print("Original observations:", len(df))
print("Hourly observations:", len(df_hourly))

print("\nSource observations per hour:")
print(df_hourly["source_observations"].value_counts())

print("\nMissing values:")
print(df_hourly.isna().sum())

print("\nHourly electricity price statistics:")
print(df_hourly["price_eur_mwh"].describe())

display(df_hourly.head(10))

Original observations: 15387
Hourly observations: 8760

Source observations per hour:
source_observations
1    6551
4    2209
Name: count, dtype: int64

Missing values:
timestamp_utc          0
price_eur_mwh          0
source_observations    0
timestamp_local        0
price_eur_kwh          0
dtype: int64

Hourly electricity price statistics:
count    8760.000000
mean       89.321723
std        52.101536
min      -250.320000
25%        71.087500
50%        92.430000
75%       114.142500
max       583.400000
Name: price_eur_mwh, dtype: float64


,timestamp_utc,price_eur_mwh,source_observations,timestamp_local,price_eur_kwh
0,2024-12-31 23:00:00+00:00,2.16,1,2025-01-01 00:00:00+01:00,0.00216
1,2025-01-01 00:00:00+00:00,1.60,1,2025-01-01 01:00:00+01:00,0.00160
2,2025-01-01 01:00:00+00:00,0.00,1,2025-01-01 02:00:00+01:00,0.00000
3,2025-01-01 02:00:00+00:00,-0.01,1,2025-01-01 03:00:00+01:00,-0.00001
4,2025-01-01 03:00:00+00:00,-0.01,1,2025-01-01 04:00:00+01:00,-0.00001
5,2025-01-01 04:00:00+00:00,-0.06,1,2025-01-01 05:00:00+01:00,-0.00006
6,2025-01-01 05:00:00+00:00,-0.09,1,2025-01-01 06:00:00+01:00,-0.00009
7,2025-01-01 06:00:00+00:00,-0.01,1,2025-01-01 07:00:00+01:00,-0.00001
8,2025-01-01 07:00:00+00:00,-0.01,1,2025-01-01 08:00:00+01:00,-0.00001
9,2025-01-01 08:00:00+00:00,-0.04,1,2025-01-01 09:00:00+01:00,-0.00004


### 7. Investigation of Validation Discrepancy

The harmonised dataset reproduces six of the seven
official annual electricity market indicators at their
published precision.

However, the calculated number of negative-price hours
is 576, compared with 573 reported by the Bundesnetzagentur.

The discrepancy is investigated by examining numerical
precision, near-zero prices and quarterly distributions.

Original electricity price observations are preserved
without modification.

In [19]:
# Investigate the effect of rounding
prices = df_hourly["price_eur_mwh"]

print("\n1. Effect of rounding")

for decimals in [6, 4, 3, 2, 1]:
    negative_hours = (prices.round(decimals) < 0).sum()
    print(f"Rounded to {decimals} decimals: {negative_hours}")

print("\n2. Near-zero negative prices")

near_zero = df_hourly[
    (prices < 0) & (prices >= -0.01)
]

display(near_zero)

print("\n3. Negative-price hours by quarter")

quarterly_validation = (
    df_hourly
    .assign(
        quarter=df_hourly["timestamp_local"].dt.quarter,
        negative_price=prices < 0
    )
    .groupby("quarter")
    .agg(
        total_hours=("price_eur_mwh", "count"),
        negative_hours=("negative_price", "sum"),
        average_price=("price_eur_mwh", "mean")
    )
)

display(quarterly_validation.round(4))


1. Effect of rounding
Rounded to 6 decimals: 576
Rounded to 4 decimals: 576
Rounded to 3 decimals: 576
Rounded to 2 decimals: 573
Rounded to 1 decimals: 477

2. Near-zero negative prices


,timestamp_utc,price_eur_mwh,source_observations,timestamp_local,price_eur_kwh
3,2025-01-01 02:00:00+00:00,-0.0100,1,2025-01-01 03:00:00+01:00,-0.000010
4,2025-01-01 03:00:00+00:00,-0.0100,1,2025-01-01 04:00:00+01:00,-0.000010
7,2025-01-01 06:00:00+00:00,-0.0100,1,2025-01-01 07:00:00+01:00,-0.000010
8,2025-01-01 07:00:00+00:00,-0.0100,1,2025-01-01 08:00:00+01:00,-0.000010
15,2025-01-01 14:00:00+00:00,-0.0100,1,2025-01-01 15:00:00+01:00,-0.000010
...,...,...,...,...,...
7130,2025-10-25 01:00:00+00:00,-0.0025,4,2025-10-25 03:00:00+02:00,-0.000003
7131,2025-10-25 02:00:00+00:00,-0.0075,4,2025-10-25 04:00:00+02:00,-0.000007
7160,2025-10-26 07:00:00+00:00,-0.0100,4,2025-10-26 08:00:00+01:00,-0.000010
7179,2025-10-27 02:00:00+00:00,-0.0025,4,2025-10-27 03:00:00+01:00,-0.000003



3. Negative-price hours by quarter


,total_hours,negative_hours,average_price
quarter,,,
1,2159,44,111.9360
2,2184,345,69.7274
3,2208,136,82.7574
4,2209,51,93.1532


### 8. Final Data Validation

The hourly dataset reproduces all seven official annual
market benchmarks when applying the relevant reporting
precision.

The initial discrepancy in negative-price hours
(576 versus 573) disappears when hourly prices are
rounded to two decimal places.

This indicates that numerical precision can account
for the observed discrepancy, although the precise
rounding convention used by SMARD has not been
independently established.

Full numerical precision is retained for all subsequent
economic analyses and charging simulations.

In [20]:
# Preserve the original validation results
validation_final = validation.copy()

# Calculate negative-price hours at reporting precision
negative_hours_rounded = (
    df_hourly["price_eur_mwh"].round(2) < 0
).sum()

# Update the benchmark comparison only
validation_final.loc[
    "Negative price hours", "Calculated"
] = negative_hours_rounded

# Recalculate differences at reporting precision
validation_final["Difference"] = (
    validation_final["Calculated"].round(2)
    - validation_final["Official SMARD"].round(2)
)

# Check whether benchmarks match
validation_final["Passed"] = np.isclose(
    validation_final["Difference"],
    0
)

display(validation_final.round(2))

# Verify structural data integrity
assert len(df_hourly) == 8760
assert df_hourly.isna().sum().sum() == 0
assert df_hourly["timestamp_utc"].is_unique
assert df_hourly["timestamp_utc"].is_monotonic_increasing
assert np.isfinite(df_hourly["price_eur_mwh"]).all()

# Verify external benchmarks
assert validation_final["Passed"].all()

print("All data validation checks passed.")

,Calculated,Official SMARD,Difference,Passed
Total hours,8760.00,8760.00,0.0,True
Average price (EUR/MWh),89.32,89.32,0.0,True
Minimum price (EUR/MWh),-250.32,-250.32,0.0,True
Maximum price (EUR/MWh),583.40,583.40,0.0,True
Negative price hours,573.00,573.00,0.0,True
Hours above 100 EUR/MWh,3494.00,3494.00,0.0,True
Hours above 300 EUR/MWh,40.00,40.00,0.0,True


All data validation checks passed.


### 9. External Data Validation

The harmonised electricity price dataset is compared
against official annual electricity market statistics
published by the Bundesnetzagentur (2026).

This validation assesses whether the retrieved and
processed data reproduce the published market indicators.

Potential discrepancies are investigated rather than
automatically corrected, since aggregation methodologies
and subsequent data revisions may differ across sources.

Reference:
Bundesnetzagentur (2026). The electricity market in 2025.
SMARD Electricity Market Data.

In [16]:
# Calculate annual statistics from hourly data
hourly_prices = df_hourly["price_eur_mwh"]

calculated_stats = {
    "Total hours": len(df_hourly),
    "Average price (EUR/MWh)": hourly_prices.mean(),
    "Minimum price (EUR/MWh)": hourly_prices.min(),
    "Maximum price (EUR/MWh)": hourly_prices.max(),
    "Negative price hours": (hourly_prices < 0).sum(),
    "Hours above 100 EUR/MWh": (hourly_prices > 100).sum(),
    "Hours above 300 EUR/MWh": (hourly_prices > 300).sum()
}

# Official Bundesnetzagentur statistics for 2025
official_stats = {
    "Total hours": 8760,
    "Average price (EUR/MWh)": 89.32,
    "Minimum price (EUR/MWh)": -250.32,
    "Maximum price (EUR/MWh)": 583.40,
    "Negative price hours": 573,
    "Hours above 100 EUR/MWh": 3494,
    "Hours above 300 EUR/MWh": 40
}

# Create validation table
validation = pd.DataFrame({
    "Calculated": calculated_stats,
    "Official SMARD": official_stats
})

# Calculate differences
validation["Difference"] = (
    validation["Calculated"]
    - validation["Official SMARD"]
)

display(validation.round(4))

,Calculated,Official SMARD,Difference
Total hours,8760.0000,8760.00,0.0000
Average price (EUR/MWh),89.3217,89.32,0.0017
Minimum price (EUR/MWh),-250.3200,-250.32,0.0000
Maximum price (EUR/MWh),583.4000,583.40,0.0000
Negative price hours,576.0000,573.00,3.0000
Hours above 100 EUR/MWh,3494.0000,3494.00,0.0000
Hours above 300 EUR/MWh,40.0000,40.00,0.0000


### 10. Export of Validated Dataset

The validated hourly electricity price dataset is
exported to the processed data directory.

All subsequent charging simulations use this common
dataset to ensure methodological consistency and
reproducibility.

The original API response remains unchanged in the
raw data directory.

In [21]:
# Define output directories
processed_dir = project_root / "data" / "processed"
tables_dir = project_root / "outputs" / "tables"

processed_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# Export hourly electricity prices
output_path = (
    processed_dir / "DE_LU_day_ahead_2025_hourly.csv"
)

df_hourly.to_csv(output_path, index=False)

# Export validation results
validation_path = (
    tables_dir / "data_validation_2025.csv"
)

validation_final.to_csv(validation_path)

print("Hourly dataset saved to:")
print(output_path)

print("\nValidation results saved to:")
print(validation_path)

print("\nTotal hourly observations:", len(df_hourly))

Hourly dataset saved to:
/Users/laramileto/Documents/HSG/5° semestre /Ökonomie der Elektromobilität/reflection paper/Economic-Value-of-Smart-EV-Charging/data/processed/DE_LU_day_ahead_2025_hourly.csv

Validation results saved to:
/Users/laramileto/Documents/HSG/5° semestre /Ökonomie der Elektromobilität/reflection paper/Economic-Value-of-Smart-EV-Charging/outputs/tables/data_validation_2025.csv

Total hourly observations: 8760
